## Разрешение пространств имён и правило LEGB (Local, Enclosing, Global, Built-in)

In [6]:
print(str)

str = "global"

def outer():
    str = "enclosing"

    def inner():
        str = "local"
        print(str)

    inner()
    print(str)

outer()
print(str)

import builtins
str = builtins.str

<class 'str'>
local
enclosing
global


## Оператор global и nonlocal

In [1]:
a = 0
b = 0

def outer():
    a = 1
    b = 1

    def inner():
        global a
        # Изменение глобальной переменной a
        a = 2
        # Изменение внешней переменной b
        nonlocal b
        b = 2

    inner()
    # Переменная a осталась прежняя, b - изменена
    print(f"a = {a} b = {b}")

outer()
# Переменная a изменена, b - осталась прежняя
print(f"a = {a} b = {b}")

a = 1 b = 2
a = 2 b = 0


## Замыкание
Функция, которая запоминает значения переменных из внешней области видимости, даже после завершения работы внешней функции

In [ ]:
# Пример замыкания для создания функции сложения с запомненным значением
def add_number(a):
    def add(b):
        return a + b

    return add

# Вызывается функция add_number, в которую передаётся значение 5 в переменную a, а затем возвращается внутренняя функция add
add_five = add_number(5)
# При вызове функции add_five (которая указывает на фукцию add) с аргументом b,
# то к этому аргументу будет прибавляться запомненное ранее значение 5
print(add_five(2))  # Выведет 7
print(add_five(10)) # Выведет 15

7
15


In [ ]:
# Пример замыкания для создания счётчика
def make_counter():
    count = 0  # <-- внешняя переменная

    def inner():
        nonlocal count
        count += 1
        return count

    return inner

# make_counter() уже отработала и переменной count в программе больше нет
counter = make_counter()

# inner() запомнила переменную count — она хранится в замыкании
print(counter())  # Выведет 1
print(counter())  # Выведет 2
print(counter())  # Выведет 3

# кортеж ячеек (cell), внутри которых Python хранит захваченные переменные
print(counter.__closure__)

1
1
1
(<cell at 0x107df6b00: int object at 0x100f32cf0>,)


## Принятие аргументов в функцию

In [ ]:
a = [1, 2, 3]

def some(arg):
    # Проверка, ссылается ли аргумент внутри функции на объект снаружи
    print(arg is a)
    arg.append(4)
    return arg

## Список в функцию передался по ссылке, а не по значению (функция может изменить оригинал!)
print(some(a))

True
[1, 2, 3, 4]
True
[1, 2, 3, 4, 4]


In [5]:
# Как неправильно:
def some(a=[]):
    # Когда функция создаётся, python вычисляет значение по умолчанию и кладёт в память
    # Далее эти же значения будут использоваться каждый раз при вызове этой же функции, а не создаваться новые
    a.append(1)
    return a

print(some())
print(some([]))
# На этом момент происходит добавление элементу к предыдущему значению списка (Неправильно!)
print(some())
print(some([]))

# Как правильно:
def some(a=None):
    if a is None:
        a = []
    a.append(1)
    return a

print(some())
print(some([]))
print(some())
print(some([]))

[1]
[1]
[1, 1]
[1]
[1]
[1]
[1]
[1]


## *args и  *kwargs

In [17]:
def some(*args, **kwargs):
    # *args - позиционные неименованные аргументы (кортеж)
    # **kwargs - именованные аргументы в виде ключ-значение (словарь)
    print(locals())

some(1, 2, 3, a = 1, b = 2)

{'args': (1, 2, 3), 'kwargs': {'a': 1, 'b': 2}}


## Использование lambda

In [20]:
a = {
    1: 3,
    2: 2,
    3: 1
}

# В качестве критерия сравнения элементов передаётся lambda с аргументов в виде ключа словаря и возвращающая значение по этому ключу
print(max(a, key=lambda x: a[x]))

1


## Встроенные функции высшего порядка: map и filter

In [ ]:
a = [ i for i in range(5) ]

# Функция map создаёт генератор
a_map = map(lambda x: x**2, a)
print(a_map)
# Формируем список
print(list(a_map))

# Функция filter создаёт генератор
a_filter = filter(lambda x: x % 2 == 0, a)
print(a_filter)
# Формируем список
print(list(a_filter))

[0, 1, 4, 9, 16]
[0, 2, 4]


## Декоратор

In [ ]:
def decorator(func):
    # Замыкание, принимающее функцию
    def wrapper(*args, **kwargs):
        # Функция обёртка пробрасывает аргументы в исходную функцию
        # Выполняет исходную функцию, а также действия до неё и после, возвращает результат
        print("BEFORE")
        result = func(*args, *kwargs)
        print("AFTER")
        return result

    return wrapper

@decorator
def say_hi(name):
    print(f"Hi, my name is {name}")
    return 1

say_hi('John')

BEFORE
Hi, my name is John
AFTER


## Декоратор с аргументами

In [ ]:
def repeat(n):
    # Дополнительный внешний слой для аргументов декоратора
    def decorator(func):
        # Замыкание, принимающее функцию
        def wrapper(*args, **kwargs):
            # Функция обёртка пробрасывает аргументы в исходную функцию
            print("BEFORE")
            result = []
            for _ in range(n):
                r = func(*args, *kwargs)
                result.append(r)
            print("AFTER")
            return result

        return wrapper
    return decorator

@repeat(3)
def say_hi(name):
    print(f"Hi, my name is {name}")
    return 1

say_hi("Alex")

BEFORE
Hi, my name is Alex
Hi, my name is Alex
Hi, my name is Alex
AFTER


[1, 1, 1]

## Итерируемый (итерабельный) объект

Итерируемым объектом в Python называется любой объект, имеющий методы `__iter__` или `__getitem__`, которые возвращают итераторы или могут принимать индексы. В итоге итерируемый объект — это объект, который может предоставить нам итератор

In [177]:
# Объект предоставляет возможность поочерёдного прохода по своим элементам
a = [1, 2, 3]

# Имеет метод __iter__, с помощью которого может быть обращён в итератор
print(hasattr(a, '__iter__'))  # True
print(hasattr(a, '__getitem__'))  # True
print(hasattr(a, '__next__'))  # False

True
True
False


## Итератор

Итератор в Python — объект, реализующий метод `__next__`, который возвращает следующий элемент последовательности или выбрасывает исключение StopIteration, если не осталось элементов.

Многие встроенные стандартные функции в Python — это итераторы: enumerate, zip, map, open (открывает файловый дескриптор на чтение построчно)

In [178]:
# Объект, который запоминает текущее положение и умеет выдавать элементы один за другим
a = [1, 2, 3]

i = iter(a)
# Имеет метод __next__, с помощью которого можно брать некий следующий элемент
print(hasattr(i, '__iter__'))  # True
print(hasattr(a, '__getitem__'))  # True
print(hasattr(i, '__next__'))  # True

print(next(i))
print(next(i))

True
True
True
1
2


## Класс итератора

In [3]:
class Range:
  """
  Объект этого класса итерабельный, так как внутри реализован магический метод __iter__,
  который отдаёт итерирующий объект (итератор)
  """
  def __init__(self, stop_value: int):
    self.current = -1
    self.stop_value = stop_value - 1

  def __iter__(self):
    """
    Возвращает итерирующий объект (итератор) для данного объекта
    """
    return RangeIterator(self)

class RangeIterator:
  """
  Объект этого класса является итератором.
  Любой итератор должен реализовывать __next__,
  в которой он должен отдавать новые значения для объектов вызывающего класса,
  пока не достигнет конца диапазона и не возбудит исключение StopIteration.
  """
  def __init__(self, container):
    self.container = container

  def __next__(self):
    """
    Возвращает следующее значение в диапазоне
    или возбуждает исключение StopIteration, если достигнут конец диапазона
    """
    if self.container.current < self.container.stop_value:
      self.container.current += 1
      return self.container.current
    raise StopIteration

# Пример использования итерируемого объекта в цикле for
_range = Range(5)
for i in _range:
    print(i, end=' ')


class Range:
    """
    В Python есть возможность объявить объекты класса и итерабельными, и итераторами одновременно.
    Для этого внутри класса нужно реализовать методы __iter__ и __next__.
    """
    def __init__(self, stop_value: int):
        self.current = -1
        self.stop_value = stop_value - 1

    def __iter__(self):
        """
        Возвращает итерирующий объект (итератор) для данного объекта
        """
        return self

    def __next__(self):
        """
        Возвращает следующее значение в диапазоне
        или возбуждает исключение StopIteration, если достигнут конец диапазона
        """
        if self.current < self.stop_value:
            self.current += 1
            return self.current
        raise StopIteration

# Цикл for под капотом использует методы __iter__ и __next__ автоматически
# Пример использования итератора вручную
iterable = Range(5)
iterator = iter(iterable)
while True:
    try:
        value = next(iterator)
        print(value, end=' ')
    except StopIteration:
        break

0 1 2 3 4 0 1 2 3 4 

In [ ]:
class Counter:
    current_value: int

    def __init__(self):
        self.current_value = 0

    def __iter__(self):
        return self

    def __next__(self):
        current_value = self.current_value
        self.current_value += 1
        return current_value

c = Counter()

i = iter(c)
print(next(i))
print(next(i))

0
1


## Генератор

Генератор — это функция, которая умеет возвращать несколько значений по очереди, не храня в памяти весь набор значений. 

В объекте-генераторе определены методы `__next__` и `__iter__`, то есть реализован протокол итератора. С этой точки зрения, в Python любой генератор является итератором.

In [ ]:
def gen_range(stop_value):
    """Генератор, который имитирует поведение функции range(stop_value)"""
    stop_value = stop_value - 1
    current = -1
    while current < stop_value:
        current += 1
        yield current

for x in gen_range(5):
    print(x, end=' ')

<generator object gen_range at 0x10bcc5b40>
0 1 2 3 4 

In [9]:
def some():
    yield 1

# При вызове функции получаем объект генератора (generator object)
obj = some()
print(obj)
# Из объекта генератора можно по требованию вызывать функцию и забирать следующее вычисленное значение
print(next(obj))
# Если объектов не осталось, то будет ошибка StopIteration
# print(next(obj))

def some():
    # При указании несколько yield генератор запоминает место, после какого yield он возвратил значение
    # и затем после следующего вызова продолжит с этого места
    print("First yield")
    yield 1
    print("Second yield")
    yield 2
    print("Third yield")
    yield 3

obj = some()
print("Start test yield")
print(next(obj))
print(next(obj))
print(next(obj))
print("Finish test yield")

def new():
    # Использование yield from позволяет делегировать часть генерации другому генератору
    print("Before yield from")
    yield from some()
    print("After yield from")
    yield 4
    yield 5

obj = new()
for o in obj:
    print(o)

<generator object some at 0x10bca8880>
1
Start test yield
First yield
1
Second yield
2
Third yield
3
Finish test yield
Before yield from
First yield
1
Second yield
2
Third yield
3
After yield from
4
5


## Конструкция yield from

Позволяет делегировать часть генерации другому итерируемому объекту или генератору

In [9]:
numbers = [1, 2, 3]
def func_simple():
    for item in numbers:
          yield item

for item in func_simple():
  print(item)

def func():
    # yield from позволяет делегировать часть генерации другому итерируемому объекту или генератору
    yield from numbers        # делегирование генерации списку numbers
    yield from func_simple()  # делегирование генерации функции func_simple()

for item in func():
  print(item, end=' ')

1
2
3
1 2 3 1 2 3 

## Генератор comprehensions

In [122]:
gen = ( i for i in range(5) if i % 2 == 0 )
print(gen)
print(next(gen))

for n in gen:
    print(n)

<generator object <genexpr> at 0x104b7cdd0>
0
2
4


## Корутины (сопрограммы)

Корутиной в Python называется функция-генератор, выполнение которой можно приостанавливать и возобновлять, при этом она может принимать данные извне в момент возобновления.

До Python 3.5 корутины реализовывались с помощью генераторов (generator-based coroutines), которые могли приостанавливать выполнение в точке yield, ожидать значение, принимать его через метод send() и затем возобновлять выполнение с места остановки.

In [5]:
def coro():
    # Ожидает значение, отправленное внутрь генератора
    value = yield
    print(value)

print(coro())
c = coro()  # Объявляем корутину
next(c)     # Просим выполнить метод до первого `yield`
c.send(10)  # Передаём данные в корутину
c.close()   # Завершаем корутину (в момент завершения генератора Python автоматически вызывает StopIteration)

<generator object coro at 0x106432c80>
10


StopIteration: 

In [ ]:
def coro():
    # Ожидает значение, отправленное внутрь корутину
    value = yield
    # Возвращает значение, умноженное на 2
    yield value * 2

# Создаём корутину
gen = coro()
# Инициализирует корутину, выполняя код до первого yield
next(gen)
# Отправляет значение внутрь корутины и получает результат выполнения второго yield
print(gen.send(10))

20


In [ ]:
def coro_with_exception():
    # Обрабатывает исключение, отправленное внутрь корутины
    try:
        yield
    except ValueError:
        yield "Исключение ValueError поймано"

# Создаём корутину
c = coro_with_exception()
# Инициализирует корутину, выполняя код до первого yield
next(c)
# Отправляет ошибку внутрь корутины
print(c.throw(ValueError))

def coro_to_close():
    # Обрабатывает закрытие корутины
    try:
        yield 1
    except GeneratorExit:
        print("Генератор закрывается")

# Создаём корутину
c = coro_to_close()
# Инициализирует корутину, выполняя код до первого yield
next(c)
# Заканчивает итератор, вызываю ошибку GeneratorExit
c.close()

Исключение ValueError поймано
Генератор закрывается


In [ ]:
import math

def cash_return_coro(percent, years):
    """Корутина для расчёта суммы вклада с учётом процентов за указанный срок"""
    # Вычисляем только 1 раз множитель для расчёта итоговой суммы
    value = math.pow(1 + percent / 100, years)
    # Запускаем бесконечный цикл для обработки входящих значений
    while True:
        try:
            # Ожидаем значение депозита, отправленное внутрь корутины
            deposit = yield
            # Вычисляем и возвращаем итоговую сумму с учётом процентов
            yield round(deposit * value, 2)
        except GeneratorExit:
            print('Выход из корутины')
            raise

# Создаём корутину
coro = cash_return_coro(5, 5)
# Инициализирует корутину, выполняя код до первого yield
next(coro)
# Отправляет значения внутрь корутины и получает результаты
values = [1000, 2000, 5000, 10000, 100000]
for item in values:
    print(coro.send(item))
    next(coro)
coro.close()

1276.28
2552.56
6381.41
12762.82
127628.16
Выход из корутины


 Поток — это составная часть процесса, который выполняется в операционной системе. Корутины выполняются в рамках одного потока или пула потоков. Но при этом принцип их работы похож на работу потоков: несколько потоков останавливаются и возобновляются, ждут друг друга, общаются.

## Менеджер контекста

In [3]:
# Позволяет обратиться к объекту и работать с полученными данными внутри этого контекста до выхода из него
# Если возникает ошибка, она может быть безопасно обработана
with open('data.txt', 'r') as file:
    # файл будет автоматически закрыт после выхода из блока with
    content = file.read()

print(content)

apple
banana
cherry


## Класс для менеджера контекста

In [129]:
class Connection:
    def __enter__(self):
        print("Открываю соединение при создании объекта")
        self.connection = 1
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        print("Закрываю соединение при выходе или ошибке")
        self.connection = 0

with Connection() as obj:
    print("Внутри контекста:", obj.connection)

print("После выхода:", obj.connection)

Открываю соединение при создании объекта
Внутри контекста: 1
Закрываю соединение при выходе или ошибке
После выхода: 0


## Функция с декоратором для менеджера контекста

In [131]:
from contextlib import contextmanager

@contextmanager
def connection():
    print("Открываю соединение")
    conn = {"state": 1}
    try:
        yield conn
    finally:
        print("Закрываю соединение")
        conn["state"] = 0

with connection() as conn:
    print("Внутри контекста:", conn["state"])

print("После выхода:", conn["state"])

Открываю соединение
Внутри контекста: 1
Закрываю соединение
После выхода: 0
